In [1]:
import os,math,warnings,math,pickle,random

from mpl_toolkits.mplot3d.proj3d import transform
from numpy.array_api import trunc

warnings.filterwarnings('ignore')
from collections import defaultdict

import pandas as pd
import numpy as np
from tqdm import tqdm

# 用于向量检索的库
import faiss

from sklearn.preprocessing import MinMaxScaler,LabelEncoder
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow.keras as keras

C:\Users\ROG\AppData\Local\Temp\ipykernel_38872\742385918.py:3: UserWarning: The numpy.array_api submodule is still experimental. See NEP 47.
  from numpy.array_api import trunc


# 1.读取数据

In [2]:
def get_all_click_sample(sample_nums=10000):
    '''debug模式，从训练集中抽取一部分数据调试代码'''
    all_click=pd.read_csv('./data/train_click_log.csv')
    all_user_ids=all_click.user_id.unique()
    sample_user_ids=np.random.choice(all_user_ids,size=sample_nums,replace=False)
    all_click=all_click[all_click['user_id'].isin(sample_user_ids)]
    all_click=all_click.drop_duplicates(['user_id','click_article_id','click_timestamp'])
    return all_click

def get_all_click_df(offline):
    '''读取点击数据，分为线上和线下，线下用训练集数据，线上用训练集+测试集'''
    if offline:
        all_click=pd.read_csv('./data/train_click_log.csv')
    else:
        trn_click=pd.read_csv('./data/train_click_log.csv')
        tst_click=pd.read_csv('./data/testA_click_log.csv')
        all_click=pd.concat([trn_click,tst_click]).reset_index(drop=True)
    all_click=all_click.drop_duplicates(['user_id','click_article_id','click_timestamp'])
    return all_click

In [3]:
def get_item_info_df():
    '''读取文章的基本属性'''
    item_info_df=pd.read_csv('./data/articles.csv')
    item_info_df=item_info_df.rename(columns={'article_id':'click_article_id'})
    return item_info_df

In [4]:
def get_item_emb_dict():
    '''读取文章的embedding'''
    item_emb_df=pd.read_csv('./data/articles_emb.csv')
    item_emb_cols=[x for x in item_emb_df.columns if 'emb' in x]
    # 把embedding转换成numpy数组，ascontiguousarray确保在内存中连续
    item_emb_np=np.ascontiguousarray(item_emb_df[item_emb_cols])
    # L2归一化
    item_emb_np=item_emb_np/np.linalg.norm(item_emb_np,axis=1,keepdims=True)
    item_emb_dict=dict(zip(item_emb_df['article_id'],item_emb_np))
    pickle.dump(item_emb_dict,open('./save/item_content_emb.pkl','wb'))
    return item_emb_dict

In [5]:
all_click_df=get_all_click_sample()
# 对时间戳归一化，按照all_click_df[['click_timestamp']]取出的是DataFrame，使用apply会对列进行操作，而all_click_df['click_timestamp']是对每个元素单独操作，取最值只会取到自己
all_click_df['click_timestamp']=all_click_df[['click_timestamp']].apply(lambda x:(x-np.min(x))/(np.max(x)-np.min(x)))

In [6]:
all_click_df.head()

,user_id,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
11,199995,336476,0.000127,4,1,17,1,25,2
12,199995,283933,0.000195,4,1,17,1,25,1
13,199995,353667,0.000210,4,1,17,1,25,1
123,199961,64329,0.000353,4,1,17,1,20,5
124,199961,199198,0.000408,4,1,17,1,20,1


In [7]:
item_info_df=get_item_info_df()

In [8]:
item_emb_dict=get_item_emb_dict()

# 2.工具函数

In [9]:
def get_user_item_time(click_df):
    '''获取用户-文章-点击时间字典 {user1:{item1:time1,item2:time2}}'''
    memo={}
    for user_id,group in click_df.groupby('user_id'):
        group_sorted=group.sort_values('click_timestamp')
        item_time_dict=dict(zip(group_sorted['click_article_id'],group_sorted['click_timestamp']))
        memo[user_id]=item_time_dict
    return memo

In [10]:
def get_item_user_time(click_df):
    '''获取文章-用户-点击时间字典  {item1: {user1: time1, user2: time2...}...}'''
    memo={}
    for click_article_id,group in click_df.groupby('click_article_id'):
        group_sorted=group.sort_values('click_timestamp')
        item_time_dict=dict(zip(group_sorted['user_id'],group_sorted['click_timestamp']))
        memo[click_article_id]=item_time_dict
    return memo

In [11]:
def get_hist_and_last_click(all_click):
    '''获取当前数据的历史点击（不包括最后一次）和最后一次点击'''
    all_click = all_click.sort_values(by=['user_id', 'click_timestamp'])
    click_last_df = all_click.groupby('user_id').tail(1)

    # 如果用户只有一个点击，hist为空了，会导致训练的时候这个用户不可见，此时默认泄露一下
    def hist_func(user_df):
        if len(user_df) == 1:
            return user_df
        else:
            return user_df[:-1]

    click_hist_df = all_click.groupby('user_id').apply(hist_func).reset_index(drop=True)

    return click_hist_df, click_last_df

In [12]:
def get_item_info_dict(item_info_df):
    '''获取文章id对应的各个属性'''
    max_min_scaler=lambda x:(x-np.min(x))/(np.max(x)-np.min(x))
    item_info_df['created_at_ts']= item_info_df[['created_at_ts']].apply(max_min_scaler)
    item_type_dict=dict(zip(item_info_df['click_article_id'],item_info_df['category_id']))
    item_words_dict=dict(zip(item_info_df['click_article_id'],item_info_df['words_count']))
    item_created_time_dict=dict(zip(item_info_df['click_article_id'],item_info_df['created_at_ts']))
    return item_type_dict,item_words_dict,item_created_time_dict

In [13]:
def get_item_topk_click(click_df,k):
    '''获取点击次数最多的k个物品'''
    return click_df['click_article_id'].value_counts().index[:k]

In [14]:
def get_user_hist_item_info_dict(all_click):
    '''获取用户历史点击的文章信息'''

    # 获取用户user_id对应的历史点击文章类型的集合字典
    user_hist_item_types=all_click.groupby('user_id')['category_id'].agg(set).reset_index()
    user_hist_item_types_dict=dict(zip(user_hist_item_types['user_id'],user_hist_item_types['category_id']))
    # 获取user_id对应的用户点击文章的集合
    user_hist_item_ids_dict=all_click.groupby('user_id')['click_article_id'].agg(set).reset_index()
    user_hist_item_ids_dict=dict(zip(user_hist_item_ids_dict['user_id'],user_hist_item_ids_dict['click_article_id']))

    # 获取user_id对应的用户历史点击的文章的平均字数字典
    user_hist_item_words=all_click.groupby('user_id')['words_count'].agg("mean").reset_index()
    user_hist_item_words_dict=dict(zip(user_hist_item_words['user_id'],user_hist_item_words['words_count']))

    # 获取user_id对应的用户最后一次点击的文章的创建时间
    all_click_=all_click.sort_values('click_timestamp')
    user_last_item_created_time=all_click_.groupby('user_id')['created_at_ts'].apply(lambda x:x.iloc[-1]).reset_index()
    max_min_scaler = lambda x : (x-np.min(x))/(np.max(x)-np.min(x))
    user_last_item_created_time['created_at_ts']=user_last_item_created_time[['created_at_ts']].apply(max_min_scaler)
    user_last_item_created_time_dict=dict(zip(user_last_item_created_time['user_id'],user_last_item_created_time['created_at_ts']))

    return user_hist_item_types_dict,user_hist_item_ids_dict,user_hist_item_words_dict,user_last_item_created_time_dict

In [15]:
item_type_dict,item_words_dict,item_created_time_dict=get_item_info_dict(item_info_df)

In [16]:
# 定义一个多路召回的字典，将各路召回的结果都保存在这个字典当中
user_multi_recall_dict={'itemcf_sim_itemcf_recall':{},'embedding_sim_item_recall':{},'youtubednn_recall':{},'youtubednn_usercf_recall':{},'cold_start_recall':{}}

In [17]:
trn_hist_click_df,trn_last_click_df=get_hist_and_last_click(all_click_df)

In [18]:
trn_last_click_df

,user_id,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
1112620,0,157507,0.597140,4,1,17,1,25,2
1112103,63,205824,0.596085,4,3,2,1,25,2
1112020,67,5595,0.595958,4,3,2,1,21,5
1111806,98,211455,0.595648,4,1,17,1,25,2
1111756,103,284470,0.595531,4,1,12,1,25,2
...,...,...,...,...,...,...,...,...,...
501584,199901,236553,0.271168,4,3,2,1,8,2
1021206,199948,218028,0.549133,4,1,17,1,2,2
1043811,199951,331116,0.568751,4,1,17,1,25,2
1041660,199961,336254,0.568413,4,1,17,1,20,1


In [19]:
def metrics_recall(user_recall_items_dict,trn_last_click_df,topk=5):
    # 对召回的结果评估
    last_click_item_dict=dict(zip(trn_last_click_df['user_id'],trn_last_click_df['click_article_id']))
    user_num=len(user_recall_items_dict)
    for k in range(10,topk+1,10):
        hit_num=0
        for user ,item_list in user_recall_items_dict.items():
            tmp=[x[0] for x in user_recall_items_dict[user][:k]]
            if last_click_item_dict[user] in set(tmp):
                hit_num+=1
        hit_rate=round(hit_num*10/user_num,5)
        print(' topk: ', k, ' : ', 'hit_num: ', hit_num, 'hit_rate: ', hit_rate, 'user_num : ', user_num)

# 3.计算相似度矩阵

In [20]:
def itemcf_sim(df,item_created_time_dict):
    '''计算物品的相似度矩阵,使用关联规则考虑了1. 用户点击的时间权重 2. 用户点击的顺序权重 3. 文章创建的时间权重'''
    user_item_time_dict =get_user_item_time(df)
    # i2i_sim[i][j]统计i和j共有的受众个数
    i2i_sim=defaultdict(dict)
    # 统计每个物品受众个数
    item_cnt=defaultdict(int)
    for user,item_time_list in tqdm(user_item_time_dict.items()):
        for loc1,(i,i_click_time) in enumerate(item_time_list.items()):
            # 更新
            item_cnt[i]+=1
            for loc2,(j, j_click_time) in enumerate(item_time_list.items()):
                if i!=j:
                    # 考虑文章的正向顺序点击和反向顺序点击
                    loc_alpha=1.0 if loc2>loc1 else 0.7
                    # 位置信息的权重
                    loc_weight=loc_alpha*(0.9**(np.abs(loc2-loc1)-1))
                    # 点击时间的权重
                    click_time_weight=np.exp(0.7**np.abs(i_click_time-j_click_time))
                    # 创建时间的权重
                    created_time_weight=np.exp(0.8**np.abs(item_created_time_dict[i]-item_created_time_dict[j]))
                    i2i_sim[i].setdefault(j,0)
                    # 加权弱化：用户点击的物品越多，对每对物品的贡献就越小
                    i2i_sim[i][j]+=loc_weight*click_time_weight*created_time_weight/math.log(len(item_time_list)+1)
    i2i_sim_=i2i_sim.copy()

    for i, related_items in i2i_sim.items():
        for j, wij in related_items.items():
            i2i_sim_[i][j]=wij/math.sqrt(item_cnt[i]*item_cnt[j])
    # 保存相似度矩阵
    pickle.dump(i2i_sim_, open('./save/itemcf_i2i_sim.pkl', 'wb'))
    return i2i_sim_

In [21]:
i2i_sim=itemcf_sim(all_click_df,item_created_time_dict)

100%|██████████| 10000/10000 [00:05<00:00, 1686.22it/s]


In [22]:
all_click_df.groupby('user_id')['click_article_id'].count()

user_id
0          2
63         2
67         2
98         2
103        2
          ..
199901     6
199948     7
199951    11
199961    24
199995     7
Name: click_article_id, Length: 10000, dtype: int64

In [23]:
def get_user_activate_degree_dict(all_click_df):
    '''获取用户活跃度'''
    all_click_df_=all_click_df.groupby('user_id')['click_article_id'].count().reset_index()

    # 归一化
    mm=MinMaxScaler()
    all_click_df_['click_article_id']=mm.fit_transform(all_click_df_[['click_article_id']])

    user_activate_degree_dict = dict(zip(all_click_df_['user_id'],all_click_df_['click_article_id']))

    return user_activate_degree_dict

In [24]:
def usercf_sim(all_click_df,user_activate_degree_dict):
    '''用户相似度计算'''
    item_user_time_dict=get_item_user_time(all_click_df)
    u2u_sim=defaultdict(dict)
    user_cnt=defaultdict(int)
    for item,user_time_list in tqdm(item_user_time_dict.items()):
        for u,click_time in user_time_list.items():
            user_cnt[u]+=1
            for v,click_time in user_time_list.items():
                u2u_sim[u].setdefault(v,0)
                if u!=v:
                    activate_weight=100*0.5*(user_activate_degree_dict[u]+user_activate_degree_dict[v])
                    u2u_sim[u][v]+=activate_weight/math.log(len(user_time_list)+1)
    u2u_sim_=u2u_sim.copy()
    for u,relater_users in u2u_sim.items():
        for v,wij in relater_users.items():
            u2u_sim_[u][v]=wij/math.sqrt(user_cnt[u]*user_cnt[v])
    pickle.dump(u2u_sim_,open('./save/usercf_u2u_sim.pkl', 'wb'))
    return u2u_sim_

In [25]:
user_activate_degree_dict=get_user_activate_degree_dict(all_click_df)
u2u_sim=usercf_sim(all_click_df,user_activate_degree_dict)

100%|██████████| 6519/6519 [00:09<00:00, 706.20it/s] 


In [29]:
def embdding_sim(click_df,item_emb_df,topk):
    '''基于文章的embedding计算相似度'''

    # 文章索引与文章id的字典映射
    item_idx_2_rawid_dict=dict(zip(item_emb_df.index,item_emb_df['article_id']))

    item_emb_cols=[x for x in item_emb_df.columns if 'emb' in x]
    item_emb_np=np.ascontiguousarray(item_emb_df[item_emb_cols].values,dtype=np.float32)
    item_emb_np=item_emb_np/np.linalg.norm(item_emb_np,axis=1,keepdims=True)

    # 建立faiss索引，基于内积
    item_index=faiss.IndexFlatIP(item_emb_np.shape[1])
    # 添加向量
    item_index.add(item_emb_np)
    # 为所有物品做一次批量检索，哈走出最相似的k个物品
    sim,idx=item_index.search(item_emb_np,topk)

    item_sim_dict=defaultdict(dict)
    for target_index,sim_value_list,rele_idx_list in tqdm(zip(range(len(item_emb_np)),sim,idx),total=len(item_emb_np)):
        # sim_value_list,rele_idx_list对当前物品所对的最相似的物品及其相似度
        target_raw_id=item_idx_2_rawid_dict[target_index]
        # 首位是物品本身
        for rele_idx,sim_value in zip(rele_idx_list[1:],sim_value_list[1:]):
            rele_raw_id=item_idx_2_rawid_dict[rele_idx]
            item_sim_dict[target_raw_id][rele_raw_id]=sim_value

    pickle.dump(item_sim_dict,open('./save/emb_i2i_sim.pkl','wb'))
    return item_sim_dict

In [27]:
pd.read_csv('./data/articles_emb.csv')

,article_id,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_240,emb_241,emb_242,emb_243,emb_244,emb_245,emb_246,emb_247,emb_248,emb_249
0,0,-0.161183,-0.957233,-0.137944,0.050855,0.830055,0.901365,-0.335148,-0.559561,-0.500603,...,0.321248,0.313999,0.636412,0.169179,0.540524,-0.813182,0.286870,-0.231686,0.597416,0.409623
1,1,-0.523216,-0.974058,0.738608,0.155234,0.626294,0.485297,-0.715657,-0.897996,-0.359747,...,-0.487843,0.823124,0.412688,-0.338654,0.320786,0.588643,-0.594137,0.182828,0.397090,-0.834364
2,2,-0.619619,-0.972960,-0.207360,-0.128861,0.044748,-0.387535,-0.730477,-0.066126,-0.754899,...,0.454756,0.473184,0.377866,-0.863887,-0.383365,0.137721,-0.810877,-0.447580,0.805932,-0.285284
3,3,-0.740843,-0.975749,0.391698,0.641738,-0.268645,0.191745,-0.825593,-0.710591,-0.040099,...,0.271535,0.036040,0.480029,-0.763173,0.022627,0.565165,-0.910286,-0.537838,0.243541,-0.885329
4,4,-0.279052,-0.972315,0.685374,0.113056,0.238315,0.271913,-0.568816,0.341194,-0.600554,...,0.238286,0.809268,0.427521,-0.615932,-0.503697,0.614450,-0.917760,-0.424061,0.185484,-0.580292
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
364042,364042,-0.055038,-0.962136,0.869436,-0.071523,-0.725294,0.434320,0.198312,-0.581154,0.702346,...,-0.410550,0.564253,-0.463959,0.167907,-0.480068,0.652090,0.380880,0.433195,-0.662455,-0.222850
364043,364043,-0.136932,-0.995471,0.991298,0.031871,-0.915621,-0.658517,0.633090,-0.564356,0.676551,...,-0.681986,-0.574185,-0.536908,0.688934,0.528204,0.162435,0.940364,0.989298,-0.761595,-0.414652
364044,364044,-0.251390,-0.976243,0.586097,0.643631,-0.663359,-0.093480,0.691554,-0.588281,0.902999,...,-0.162220,-0.242030,-0.476131,0.352132,-0.311279,0.460574,-0.653077,-0.143725,0.068093,-0.705010
364045,364045,0.224342,-0.923288,-0.381742,0.687890,-0.773911,-0.103629,-0.406486,0.246004,0.255191,...,-0.422999,0.390324,0.655911,-0.646753,-0.174031,0.698037,-0.317102,0.687132,-0.531512,0.010726


In [28]:
item_emb_dict

{0: array([-0.02118425, -0.12580893, -0.01813001,  0.00668391,  0.10909397,
         0.11846624, -0.04404838, -0.07354293, -0.06579412,  0.02170995,
         0.05630901,  0.04666488,  0.11492702, -0.06951095,  0.08220764,
         0.03534407, -0.10814503, -0.09250724, -0.08225472, -0.02008969,
        -0.08756393,  0.00569023,  0.02347829,  0.00616275,  0.07813909,
        -0.02409734,  0.02564285, -0.06146177, -0.04006071,  0.04641773,
         0.03656221,  0.07079111, -0.04878796,  0.06438719, -0.01364673,
         0.01566297,  0.01740611, -0.08162891, -0.0595786 ,  0.04555704,
        -0.00811461, -0.09602179, -0.05048424, -0.12364366,  0.00806219,
         0.06342559,  0.038073  , -0.08184084, -0.00657207,  0.05539924,
        -0.03188176,  0.08788847, -0.06689828, -0.06069421,  0.00577   ,
         0.03791584,  0.05912034, -0.03743939,  0.12048548,  0.09241205,
         0.11193531, -0.08243855,  0.04701659,  0.0512825 ,  0.08581513,
         0.01362305,  0.10491944, -0.01347765, -

In [30]:
item_emb_df=pd.read_csv('./data/articles_emb.csv').sample(10000,random_state=0).reset_index(drop=True)
emb_i2i_sim=embdding_sim(all_click_df,item_emb_df,topk=10)

100%|██████████| 10000/10000 [00:00<00:00, 165807.02it/s]


# 4.召回

## youtubeDnn召回

In [ ]:
def gen_data_set(data,negsample=0):
    '''获取youtubeDnn召回召回时的训练和验证数据，这里进行负采样'''
    data.sort_values('click_timestamp',inplace=True)
    item_ids=data['click_article_id'].unique()

    # 数据的格式 [user_id, 交互过的历史物品, 下一个要交互的物品, 正/负本, 历史行为的长度]
    train_set=[]
    test_set=[]

    for reviewerID, hist in tqdm(data.groupby('user_id')):
        pos_list=hist['click_article_id'].tolist()

        # 用户交互过的物品只有一个也要放到训练集中，否则会造成embedding缺失
        if len(pos_list)==1:
            train_set.append((reviewerID,[pos_list[0]],pos_list[0],1,len(pos_list)))
            test_set.append((reviewerID,[pos_list[0]],pos_list[0],1,len(pos_list)))

        if negsample>0:
            candidate_set=list(set(item_ids)-set(pos_list))
            neg_list=np.random.choice(candidate_set,size=len(pos_list)*negsample,replace=False)# 为每个正样本抽取n个负样本，replace控制是否可以放回抽样
        for i in range(1,len(pos_list)):
            hist=pos_list[:i] # 历史物品
            if i!=len(pos_list)-1:
                train_set.append((reviewerID,hist[::-1],pos_list[i],1,len(hist)))
                for negi in range(negsample):
                    train_set.append((reviewerID,hist[::-1],neg_list[i*negsample+negi],0,len(hist)))
            else:
                # 最长的序列作为测试数据
                test_set.append((reviewerID,hist[::-1],pos_list[i],1,len(hist)))
    random.shuffle(train_set)
    random.shuffle(test_set)
    return train_set,test_set